# Torque Sensor Data Analysis

Load and visualize CSV data from the Torque Cell DAQ system (DYJN-104 + NI USB-6002).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path
import numpy as np

# Use dark style to match the DAQ dashboard theme
plt.style.use("dark_background")
plt.rcParams.update({
    "figure.figsize": (12, 5),
    "figure.dpi": 100,
    "axes.grid": True,
    "grid.alpha": 0.2,
    "font.size": 11,
})

# Color palette (Catppuccin Mocha)
BLUE = "#89b4fa"
GREEN = "#a6e3a1"
RED = "#f38ba8"
YELLOW = "#f9e2af"
PEACH = "#fab387"
MAUVE = "#cba6f7"

## 1. Load CSV Data

Select which CSV file to analyze. The loader skips the metadata comment lines automatically.

In [ ]:
# List available data files
data_dir = Path("../data")
csv_files = sorted(data_dir.glob("torque_log_*.csv"))

print(f"Found {len(csv_files)} data files:")
for i, f in enumerate(csv_files):
    size_kb = f.stat().st_size / 1024
    print(f"  [{i}] {f.name}  ({size_kb:.1f} KB)")

In [ ]:
# Select file index (change this to load a different file)
FILE_INDEX = -1  # -1 = most recent

csv_path = csv_files[FILE_INDEX]
print(f"Loading: {csv_path.name}")

# Read metadata from comment lines
metadata = {}
with open(csv_path) as f:
    for line in f:
        if not line.startswith("#"):
            break
        key, _, val = line[2:].strip().partition(": ")
        metadata[key] = val

print("\nMetadata:")
for k, v in metadata.items():
    print(f"  {k}: {v}")

# Load data
df = pd.read_csv(csv_path, comment="#")
df["timestamp"] = pd.to_datetime(df["timestamp"])

# Replace empty torque with NaN
if "torque_Nm" in df.columns:
    df["torque_Nm"] = pd.to_numeric(df["torque_Nm"], errors="coerce")

has_torque = df["torque_Nm"].notna().any() if "torque_Nm" in df.columns else False

print(f"\nLoaded {len(df)} samples")
print(f"Duration: {df['elapsed_s'].iloc[-1]:.2f} s")
print(f"Sample rate: {metadata.get('Sample Rate', 'unknown')} Hz")
print(f"Torque calibration: {'Yes' if has_torque else 'No (raw voltage only)'}")
df.head()

## 2. Voltage vs Time

In [ ]:
fig, ax = plt.subplots()
ax.plot(df["elapsed_s"], df["voltage_V"], color=BLUE, linewidth=0.8, label="Voltage")
ax.set_xlabel("Elapsed Time (s)")
ax.set_ylabel("Voltage (V)")
ax.set_title(f"Voltage vs Time — {csv_path.name}")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Torque vs Time

Only plotted if calibration data was applied during recording. If not, you can apply a linear calibration below.

In [ ]:
# Apply post-hoc calibration if torque column is empty
# Set these values to match your DYJN-104 + 510 transmitter setup
APPLY_CALIBRATION = not has_torque  # Only apply if no calibration exists
CAL_SLOPE = 10.0    # Nm per Volt
CAL_OFFSET = 0.0    # Nm

if APPLY_CALIBRATION:
    df["torque_Nm"] = CAL_SLOPE * df["voltage_V"] + CAL_OFFSET
    has_torque = True
    print(f"Applied post-hoc calibration: torque = {CAL_SLOPE} * voltage + {CAL_OFFSET}")
else:
    print("Using calibration from recording session")

In [ ]:
if has_torque:
    fig, ax = plt.subplots()
    ax.plot(df["elapsed_s"], df["torque_Nm"], color=GREEN, linewidth=0.8, label="Torque")
    ax.set_xlabel("Elapsed Time (s)")
    ax.set_ylabel("Torque (Nm)")
    ax.set_title(f"Torque vs Time — {csv_path.name}")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No torque data available. Set APPLY_CALIBRATION = True above.")

## 4. Combined Plot (Dual Y-Axis)

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 5))

ax1.plot(df["elapsed_s"], df["voltage_V"], color=BLUE, linewidth=0.8, label="Voltage (V)")
ax1.set_xlabel("Elapsed Time (s)")
ax1.set_ylabel("Voltage (V)", color=BLUE)
ax1.tick_params(axis="y", labelcolor=BLUE)

if has_torque:
    ax2 = ax1.twinx()
    ax2.plot(df["elapsed_s"], df["torque_Nm"], color=GREEN, linewidth=0.8, label="Torque (Nm)")
    ax2.set_ylabel("Torque (Nm)", color=GREEN)
    ax2.tick_params(axis="y", labelcolor=GREEN)
    # Combined legend
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
else:
    ax1.legend()

ax1.set_title(f"Voltage & Torque — {csv_path.name}")
plt.tight_layout()
plt.show()

## 5. Statistics Summary

In [ ]:
cols = ["voltage_V"]
if has_torque:
    cols.append("torque_Nm")

stats = df[cols].describe()

# Add additional stats
for col in cols:
    stats.loc["range", col] = df[col].max() - df[col].min()
    stats.loc["median", col] = df[col].median()

# Compute actual sample rate from timestamps
dt = df["elapsed_s"].diff().dropna()
stats.loc["actual_rate_Hz", cols[0]] = 1.0 / dt.mean() if dt.mean() > 0 else 0

print(f"Recording: {csv_path.name}")
print(f"Duration:  {df['elapsed_s'].iloc[-1]:.3f} s")
print(f"Samples:   {len(df)}")
print()
stats.round(4)

## 6. Signal Distribution (Histogram)

In [ ]:
ncols = 2 if has_torque else 1
fig, axes = plt.subplots(1, ncols, figsize=(12, 4))
if ncols == 1:
    axes = [axes]

axes[0].hist(df["voltage_V"], bins=50, color=BLUE, alpha=0.85, edgecolor="none")
axes[0].set_xlabel("Voltage (V)")
axes[0].set_ylabel("Count")
axes[0].set_title("Voltage Distribution")

if has_torque:
    axes[1].hist(df["torque_Nm"], bins=50, color=GREEN, alpha=0.85, edgecolor="none")
    axes[1].set_xlabel("Torque (Nm)")
    axes[1].set_ylabel("Count")
    axes[1].set_title("Torque Distribution")

plt.tight_layout()
plt.show()

## 7. Compare Multiple Recordings

In [ ]:
colors = [BLUE, GREEN, RED, YELLOW, PEACH, MAUVE, "#74c7ec", "#94e2d5", "#f5c2e7", "#b4befe"]

fig, ax = plt.subplots(figsize=(12, 5))

for i, path in enumerate(csv_files):
    d = pd.read_csv(path, comment="#")
    if len(d) < 2:
        continue
    ax.plot(
        d["elapsed_s"], d["voltage_V"],
        color=colors[i % len(colors)],
        linewidth=0.8,
        label=path.name,
        alpha=0.85,
    )

ax.set_xlabel("Elapsed Time (s)")
ax.set_ylabel("Voltage (V)")
ax.set_title("All Recordings — Voltage Overlay")
ax.legend(fontsize=8, loc="best")
plt.tight_layout()
plt.show()